In [0]:
pip install selenium webdriver-manager

In [0]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Create a driver object
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# Now you can access capabilities
print(driver.capabilities)

driver.quit()

In [0]:
import time
import logging
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from selenium.common.exceptions import (
    NoSuchElementException,
    TimeoutException,
    ElementClickInterceptedException
)
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

# ─────────────────────────────────────────────
# CONFIGURATION — Edit these values
# ─────────────────────────────────────────────
CONFIG = {
    "email":           "vaidyaprakhar5@gmail.com",
    "password":        "Prak#0651",
    "job_title":       "Data Engineer",       # Job keyword
    "location":        "India",           # Job location
    "phone_number":    "9555406367",              # Your phone number
    "resume_path":     "",# Absolute path to resume
    "max_jobs":        1,                         # Max number of jobs to apply
    "experience_years":"7",                        # Years of experience
    "cover_letter":    "I am excited to apply for this position and believe my skills are a strong match.", 
}

# ─────────────────────────────────────────────
# LOGGING SETUP
# ─────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("linkedin_apply.log"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)


# ─────────────────────────────────────────────
# DRIVER SETUP
# ─────────────────────────────────────────────
def init_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-popup-blocking")
    # options.add_argument("--headless")  # Uncomment to run headless
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.implicitly_wait(5)
    return driver


# ─────────────────────────────────────────────
# LOGIN
# ─────────────────────────────────────────────
def login(driver):
    log.info("Logging into LinkedIn...")
    driver.get("https://www.linkedin.com/login")
    wait = WebDriverWait(driver, 15)

    wait.until(EC.presence_of_element_located((By.ID, "username"))).send_keys(CONFIG["email"])
    driver.find_element(By.ID, "password").send_keys(CONFIG["password"])
    driver.find_element(By.XPATH, '//button[@type="submit"]').click()
    time.sleep(4)

    if "feed" in driver.current_url or "mynetwork" in driver.current_url:
        log.info("Login successful!")
    else:
        log.warning("Login may have failed or requires CAPTCHA. Please check the browser.")
        input(">>> Press ENTER after completing any manual verification to continue...")


# ─────────────────────────────────────────────
# SEARCH EASY APPLY JOBS
# ─────────────────────────────────────────────
def search_jobs(driver):
    log.info(f"Searching for '{CONFIG['job_title']}' jobs in '{CONFIG['location']}'...")
    search_url = (
        f"https://www.linkedin.com/jobs/search/"
        f"?keywords={CONFIG['job_title'].replace(' ', '%20')}"
        f"&location={CONFIG['location'].replace(' ', '%20')}"
        f"&f_AL=true"   # f_AL=true filters Easy Apply jobs only
        f"&sortBy=DD"   # Sort by most recent
    )
    driver.get(search_url)
    time.sleep(3)
    log.info("Job search results loaded with Easy Apply filter.")


# ─────────────────────────────────────────────
# SCROLL TO LOAD MORE JOBS
# ─────────────────────────────────────────────
def scroll_jobs_panel(driver):
    try:
        job_list_container = driver.find_element(
            By.CLASS_NAME, "jobs-search-results-list"
        )
        driver.execute_script(
            "arguments[0].scrollTop = arguments[0].scrollHeight", job_list_container
        )
        time.sleep(2)
    except Exception:
        pass


# ─────────────────────────────────────────────
# HANDLE EASY APPLY FORM FIELDS
# ─────────────────────────────────────────────
def fill_form_fields(driver):
    """Fill common Easy Apply form fields."""
    wait = WebDriverWait(driver, 5)

    # --- Phone Number ---
    try:
        phone_field = driver.find_element(
            By.XPATH, '//input[contains(@id,"phoneNumber") or contains(@name,"phoneNumber")]'
        )
        if phone_field.get_attribute("value") == "":
            phone_field.clear()
            phone_field.send_keys(CONFIG["phone_number"])
            log.info("Filled phone number.")
    except NoSuchElementException:
        pass

    # --- Resume Upload ---
    try:
        upload_btn = driver.find_element(By.XPATH, '//input[@type="file"]')
        upload_btn.send_keys(CONFIG["resume_path"])
        log.info("Uploaded resume.")
        time.sleep(2)
    except NoSuchElementException:
        pass

    # --- Text Inputs (Years of experience, etc.) ---
    try:
        text_inputs = driver.find_elements(
            By.XPATH,
            '//input[@type="text" and not(@readonly) and not(@disabled)]'
        )
        for field in text_inputs:
            if field.get_attribute("value") == "":
                label_text = ""
                try:
                    field_id = field.get_attribute("id")
                    label = driver.find_element(By.XPATH, f'//label[@for="{field_id}"]')
                    label_text = label.text.lower()
                except Exception:
                    pass

                if "year" in label_text or "experience" in label_text:
                    field.send_keys(CONFIG["experience_years"])
                    log.info(f"Filled experience field: {label_text}")
                elif "city" in label_text or "location" in label_text:
                    field.send_keys(CONFIG["location"])
    except Exception:
        pass

    # --- Dropdowns (Select elements) ---
    try:
        selects = driver.find_elements(By.TAG_NAME, "select")
        for select_el in selects:
            sel = Select(select_el)
            if len(sel.options) > 1:
                sel.select_by_index(1)  # Select first real option
                log.info("Selected dropdown option.")
    except Exception:
        pass

    # --- Radio Buttons (Yes/No questions) ---
    try:
        radios = driver.find_elements(
            By.XPATH, '//input[@type="radio"]'
        )
        for radio in radios:
            if not radio.is_selected():
                # Prefer "Yes" or first option
                try:
                    radio_label = driver.find_element(
                        By.XPATH, f'//label[@for="{radio.get_attribute("id")}"]'
                    )
                    if "yes" in radio_label.text.lower():
                        radio.click()
                        log.info("Selected 'Yes' radio button.")
                        break
                except Exception:
                    pass
    except Exception:
        pass

    # --- Textarea (Cover Letter) ---
    try:
        textareas = driver.find_elements(By.TAG_NAME, "textarea")
        for area in textareas:
            if area.get_attribute("value") == "" or area.text == "":
                area.send_keys(CONFIG["cover_letter"])
                log.info("Filled textarea/cover letter.")
    except Exception:
        pass


# ─────────────────────────────────────────────
# HANDLE MULTI-STEP EASY APPLY MODAL
# ─────────────────────────────────────────────
def handle_easy_apply_modal(driver):
    """Navigate through all steps of the Easy Apply modal."""
    wait = WebDriverWait(driver, 10)
    max_steps = 10  # Safety limit for steps
    step = 0

    while step < max_steps:
        step += 1
        time.sleep(2)
        fill_form_fields(driver)
        time.sleep(1)

        # Check for "Submit application" button → Final step
        try:
            submit_btn = driver.find_element(
                By.XPATH,
                '//button[contains(@aria-label,"Submit application") or contains(text(),"Submit application")]'
            )
            submit_btn.click()
            log.info("✅ Application SUBMITTED successfully!")
            time.sleep(2)
            return True
        except NoSuchElementException:
            pass

        # Check for "Review" button
        try:
            review_btn = driver.find_element(
                By.XPATH,
                '//button[contains(@aria-label,"Review") or contains(text(),"Review")]'
            )
            review_btn.click()
            log.info(f"Step {step}: Clicked 'Review'.")
            continue
        except NoSuchElementException:
            pass

        # Check for "Next" button → Move to next step
        try:
            next_btn = driver.find_element(
                By.XPATH,
                '//button[contains(@aria-label,"Continue to next step") or contains(text(),"Next")]'
            )
            next_btn.click()
            log.info(f"Step {step}: Clicked 'Next'.")
            continue
        except NoSuchElementException:
            pass

        # If nothing matched, break out
        log.warning(f"Step {step}: No actionable button found. Exiting modal.")
        break

    # Close the modal if application wasn't submitted
    try:
        close_btn = driver.find_element(
            By.XPATH, '//button[@aria-label="Dismiss" or @data-test-modal-close-btn]'
        )
        close_btn.click()
        time.sleep(1)
        # Confirm discard if prompted
        try:
            discard_btn = driver.find_element(
                By.XPATH, '//button[contains(text(),"Discard")]'
            )
            discard_btn.click()
        except Exception:
            pass
    except Exception:
        pass

    return False


# ─────────────────────────────────────────────
# MAIN APPLY LOOP
# ─────────────────────────────────────────────
def apply_to_jobs(driver):
    applied_count = 0
    skipped_count = 0
    page = 0

    while applied_count < CONFIG["max_jobs"]:
        page += 1
        log.info(f"\n--- Processing Page {page} ---")
        scroll_jobs_panel(driver)
        time.sleep(2)

        # Get all job cards
        try:
            job_cards = WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located(
                    (By.XPATH, '//li[contains(@class,"jobs-search-results__list-item")]')
                )
            )
        except TimeoutException:
            log.warning("No job cards found. Ending search.")
            break

        log.info(f"Found {len(job_cards)} job listings on this page.")

        for index, card in enumerate(job_cards):
            if applied_count >= CONFIG["max_jobs"]:
                break

            try:
                # Click the job card
                driver.execute_script("arguments[0].scrollIntoView(true);", card)
                card.click()
                time.sleep(2)

                # Get job title & company for logging
                try:
                    job_title_el = driver.find_element(
                        By.CLASS_NAME, "job-details-jobs-unified-top-card__job-title"
                    )
                    company_el = driver.find_element(
                        By.CLASS_NAME, "job-details-jobs-unified-top-card__company-name"
                    )
                    job_name = job_title_el.text.strip()
                    company_name = company_el.text.strip()
                    log.info(f"[{index+1}] {job_name} @ {company_name}")
                except Exception:
                    job_name = "Unknown"
                    company_name = "Unknown"

                # Look for Easy Apply button
                try:
                    easy_apply_btn = WebDriverWait(driver, 5).until(
                        EC.element_to_be_clickable(
                            (By.XPATH,
                             '//button[contains(@aria-label,"Easy Apply") or '
                             '(contains(@class,"jobs-apply-button") and contains(text(),"Easy Apply"))]')
                        )
                    )
                    easy_apply_btn.click()
                    log.info(f"Clicked Easy Apply for: {job_name}")
                    time.sleep(2)

                    # Handle the modal
                    success = handle_easy_apply_modal(driver)
                    if success:
                        applied_count += 1
                        log.info(f"✅ [{applied_count}/{CONFIG['max_jobs']}] Applied: {job_name} @ {company_name}")
                    else:
                        skipped_count += 1
                        log.warning(f"⚠️  Skipped (complex form): {job_name} @ {company_name}")

                except TimeoutException:
                    skipped_count += 1
                    log.info(f"⏭️  No Easy Apply button found. Skipping: {job_name}")

                time.sleep(2)

            except ElementClickInterceptedException:
                log.warning(f"Could not click job card {index+1}. Skipping.")
            except Exception as e:
                log.error(f"Unexpected error on job {index+1}: {e}")

        # Go to next page
        try:
            next_page_btn = driver.find_element(
                By.XPATH, '//button[@aria-label="View next page"]'
            )
            next_page_btn.click()
            log.info("Moving to next page...")
            time.sleep(4)
        except NoSuchElementException:
            log.info("No more pages available.")
            break

    log.info(f"\n{'='*50}")
    log.info(f"🎯 Job Application Session Complete!")
    log.info(f"   ✅ Applied  : {applied_count}")
    log.info(f"   ⏭️  Skipped  : {skipped_count}")
    log.info(f"{'='*50}\n")


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────
if __name__ == "__main__":
    driver = init_driver()
    try:
        login(driver)
        search_jobs(driver)
        apply_to_jobs(driver)
    except KeyboardInterrupt:
        log.info("Script interrupted by user.")
    except Exception as e:
        log.critical(f"Fatal error: {e}", exc_info=True)
    finally:
        driver.quit()
        log.info("Browser closed.")
